# Semana 5 — Tarea del proyecto ancla · Redes neuronales para el aviso y la sílice
### Cuaderno guiado (ALUMNO)

En el cuaderno satélite `Semana5_mlp_ALUMNO.ipynb` entrenaste perceptrones multicapa (MLP) con
Keras: los construiste, los entrenaste con parada temprana y los usaste para clasificar y para estimar un
número. Ahora los aplicas a la planta de flotación, en las dos tareas de las semanas anteriores:

- **clasificar** cada hora como dentro o fuera de especificación (sílice por encima del 3 %), como en la
  Semana 3;
- **estimar** el % de sílice de cada hora, como en la Semana 2.

**Sobre los datos.** Usamos la misma tabla horaria **semisintética** de las semanas anteriores
(`flotacion_semisintetica_horaria.csv`): las variables de proceso son las reales de la planta y la
sílice del concentrado está simulada con un modelo del proceso que no se publica. La columna
`fuera_spec` vale 1 cuando la sílice de esa hora supera el 3 %.

**Qué te damos hecho.** La sección 1 está resuelta: carga la tabla, fija las entradas, divide las horas
en tres partes respetando el tiempo, estandariza y calcula las referencias de las Semanas 2 y 3. Solo
tienes que ejecutarla y leerla.

**Qué entregas esta semana (tarea semanal)**

- Las **secciones 2 y 3**: una **red de clasificación** y una **red de regresión**, comparadas con las
  referencias. Esta vez **escribes tú el código de cada red**, en una sola celda, a partir de la
  especificación del enunciado. Responde a las tres preguntas en las celdas «Tu respuesta».
- Se entrega como archivo `.ipynb` en el Campus Virtual, ejecutado de principio a fin sin errores.

**Proyecto final.** La parte del proyecto final de esta semana llegará en un cuaderno aparte, cuando
hayamos visto las redes convolucionales y secuenciales.

**Cómo trabajar**

- Ejecuta las celdas **en orden**, de arriba abajo.
- En las celdas **«Tu código»** escribes tú todo el código. El comentario dice qué tiene que hacer, paso
  a paso, y qué variables tiene que dejar preparadas, porque las celdas siguientes las usan. Te servirá
  de modelo el código del cuaderno satélite.
- Después de cada red hay una celda **Comprueba**. La misma red da resultados algo distintos en cada
  ordenador, así que no pide cifras exactas: comprueba que la red está construida como pide el enunciado
  y que sus resultados son razonables.
- Un resultado modesto **no es un fracaso**: lo importante es que el método sea correcto y que
  interpretes el resultado con honestidad.

## 0. Preparación

Cargamos las librerías (importar TensorFlow tarda unos segundos). Ejecuta la celda sin cambiar nada.

In [ ]:
%matplotlib inline
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'   # oculta los mensajes informativos de TensorFlow

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from tensorflow import keras
from tensorflow.keras import layers

pd.set_option('display.width', 120)

## 1. Los datos (resuelto)

Esta sección ya está hecha: ejecuta sus celdas sin cambiar nada y lee los resultados. Las secciones 2
y 3 usan las variables que se definen aquí.

### 1a. Cargar la tabla

La tabla tiene **una fila por hora** (marzo–septiembre de 2017). Cada fila trae la media horaria de las
variables de proceso, la sílice de esa hora y la etiqueta `fuera_spec`.

| Columna | Qué contiene | Papel en la tarea |
|---|---|---|
| `date` | hora a la que corresponde la fila | metadato |
| `n_muestras` | número de registros de 20 s promediados en esa hora | metadato |
| `silice` | % de sílice en el concentrado (semisintética) | **salida de la red de regresión** |
| `iron_feed`, `silica_feed` | % de hierro y de sílice del mineral que entra | entrada |
| `almidon_medio` | caudal de almidón | entrada |
| `amina_media` | caudal de amina | entrada |
| `caudal_pulpa` | caudal de pulpa | entrada |
| `ph_medio` | pH de la pulpa | entrada |
| `densidad` | densidad de la pulpa | entrada |
| `aire_medio` | aire inyectado, media de las 7 columnas de flotación | entrada |
| `nivel_medio` | nivel de espuma, media de las 7 columnas | entrada |
| `fuera_spec` | 1 si la sílice de esa hora supera el 3 %, 0 si no | **salida de la red de clasificación** |

Los datos están en la carpeta `datasets`, al lado de la carpeta de esta semana. Si la celda siguiente
da `FileNotFoundError`, revisa que has descargado el CSV en `SII/datasets/`.

In [ ]:
ruta_datos = '../datasets/flotacion_semisintetica_horaria.csv'
tabla = pd.read_csv(ruta_datos, parse_dates=['date'])

n_filas = tabla.shape[0]
n_columnas = tabla.shape[1]

print("Filas:", n_filas, "| columnas:", n_columnas)
tabla.head()

Comprobamos que has cargado el fichero correcto:

In [ ]:
assert n_filas == 4097, "Deberías tener 4097 filas: ¿has cargado flotacion_semisintetica_horaria.csv?"
assert n_columnas == 13, "Deberías tener 13 columnas"
print("Correcto")

### 1b. Entradas y salidas (sin fugas)

Las dos redes usan las mismas entradas: solo las variables que la planta mide en tiempo real, las de
papel «entrada» en la tabla de 1a. Quedan **prohibidas** como entrada:

- `silice` y `fuera_spec`: son las salidas, y `fuera_spec` se calcula a partir de la sílice. Usar una
  de las dos para predecir la otra sería darle al modelo la respuesta (**fuga de datos**).
- `date` y `n_muestras`: son metadatos, no describen el proceso.

In [ ]:
objetivo_clase = 'fuera_spec'
objetivo_silice = 'silice'

variables_proceso = ['iron_feed', 'silica_feed', 'almidon_medio', 'amina_media',
                     'caudal_pulpa', 'ph_medio', 'densidad', 'aire_medio', 'nivel_medio']

print(len(variables_proceso), "entradas:", variables_proceso)

### 1c. Ajuste, validación y test, respetando el tiempo

Como en las Semanas 2 y 3, el **test** son las últimas horas (20 %), sin barajar: es el mismo test que
entonces, así que las cifras se pueden comparar. Además, como en el satélite, la red necesita una
**validación** para la parada temprana. La sacamos de las últimas horas del train (20 %), también sin
barajar: la red aprende con la parte de **ajuste** y la parada temprana mira la **validación**.

| Parte | Horas | Para qué |
|---|---|---|
| ajuste | las primeras 2622 | entrenar las redes |
| validación | las 655 siguientes | parada temprana |
| test | las últimas 820 | evaluar al final |

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `train`, `test` | las primeras 3277 horas (80 %) y las 820 restantes |
| `ajuste`, `validacion` | las primeras 2622 horas del train y sus últimas 655 |

In [ ]:
fraccion_train = 0.8
n_train = int(n_filas * fraccion_train)
train = tabla.iloc[:n_train]
test = tabla.iloc[n_train:]

fraccion_validacion = 0.2
n_validacion = int(n_train * fraccion_validacion)
n_ajuste = n_train - n_validacion
ajuste = train.iloc[:n_ajuste]
validacion = train.iloc[n_ajuste:]

print(f"ajuste    : {len(ajuste)} horas, del {ajuste['date'].min():%Y-%m-%d} al {ajuste['date'].max():%Y-%m-%d}")
print(f"validacion: {len(validacion)} horas, del {validacion['date'].min():%Y-%m-%d} al {validacion['date'].max():%Y-%m-%d}")
print(f"test      : {len(test)} horas, del {test['date'].min():%Y-%m-%d} al {test['date'].max():%Y-%m-%d}")

### 1d. Estandarizar

Las redes, como el descenso de gradiente de la Semana 2, aprenden mejor con las entradas
estandarizadas. El escalador se ajusta **solo con la parte de ajuste**, la única con la que aprenden las
redes.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `escalador` | `StandardScaler` ajustado solo con las entradas de `ajuste` |
| `X_ajuste`, `X_validacion`, `X_test` | las 9 entradas estandarizadas de cada parte |
| `y_ajuste_clase`, `y_validacion_clase`, `y_test_clase` | `fuera_spec` de cada parte |
| `y_ajuste_silice`, `y_validacion_silice`, `y_test_silice` | `silice` de cada parte |

In [ ]:
escalador = StandardScaler()
escalador.fit(ajuste[variables_proceso])   # solo con la parte de ajuste

X_ajuste = escalador.transform(ajuste[variables_proceso])
X_validacion = escalador.transform(validacion[variables_proceso])
X_test = escalador.transform(test[variables_proceso])

y_ajuste_clase = ajuste[objetivo_clase].to_numpy()
y_validacion_clase = validacion[objetivo_clase].to_numpy()
y_test_clase = test[objetivo_clase].to_numpy()

y_ajuste_silice = ajuste[objetivo_silice].to_numpy()
y_validacion_silice = validacion[objetivo_silice].to_numpy()
y_test_silice = test[objetivo_silice].to_numpy()

print("X_ajuste:", X_ajuste.shape, "| X_validacion:", X_validacion.shape, "| X_test:", X_test.shape)

### 1e. Las referencias de las Semanas 2 y 3

Para saber si las redes valen la pena, las comparamos con lo que ya tenías:

- **Aviso** (Semana 3): la línea base, que dice siempre «dentro de especificación», y la regresión
  logística.
- **Sílice** (Semana 2): la línea base, que predice siempre la media de la sílice del train, y la
  regresión lineal.

Para que salgan las mismas cifras que entonces, la logística y la lineal se entrenan como en esas
semanas: con todo el train (ajuste y validación) y con su propio escalador.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `prediccion_base_clase` | 0 (dentro de especificación) para todas las horas del test |
| `prediccion_base_silice` | la media de la sílice del train, para todas las horas del test |
| `probabilidad_logistica` | probabilidad de «fuera de especificación» que da la logística en cada hora del test |
| `prediccion_logistica` | 1 si `probabilidad_logistica` llega a 0,5; 0 si no |
| `prediccion_lineal` | sílice que estima la regresión lineal en cada hora del test |

In [ ]:
escalador_train = StandardScaler()
escalador_train.fit(train[variables_proceso])
X_train_referencias = escalador_train.transform(train[variables_proceso])
X_test_referencias = escalador_train.transform(test[variables_proceso])
y_train_clase = train[objetivo_clase].to_numpy()
y_train_silice = train[objetivo_silice].to_numpy()

n_test = len(test)
prediccion_base_clase = np.zeros(n_test, dtype=int)
media_silice_train = y_train_silice.mean()
prediccion_base_silice = np.full(n_test, media_silice_train)

logistica = LogisticRegression(max_iter=2000)
logistica.fit(X_train_referencias, y_train_clase)
probabilidades_logistica = logistica.predict_proba(X_test_referencias)
probabilidad_logistica = probabilidades_logistica[:, 1]
supera_umbral_logistica = probabilidad_logistica >= 0.5
prediccion_logistica = supera_umbral_logistica.astype(int)

lineal = LinearRegression()
lineal.fit(X_train_referencias, y_train_silice)
prediccion_lineal = lineal.predict(X_test_referencias)

print(f"Logística (Semana 3): AUC en el test = {roc_auc_score(y_test_clase, probabilidad_logistica):.3f}")
print(f"Lineal (Semana 2): MAE en el test = {mean_absolute_error(y_test_silice, prediccion_lineal):.3f} % de sílice")

**Lo que conviene recordar de los datos**

- Una de cada cuatro horas está fuera de especificación; en el test, 189 de 820 (23 %).
- Las redes aprenden con las 2622 horas de ajuste; la logística y la lineal, con las 3277 del train.
- Hay un hueco de 319 horas sin registros al principio. En la tarea no hay que hacer nada con él.

## 2. Red de clasificación: avisar de las horas fuera de especificación

En el satélite, la capa de salida de la red de dígitos tenía 10 neuronas con softmax, una por clase.
Con **dos clases** basta **una neurona con activación sigmoide**, la función logística de la Semana 3: da
la probabilidad de la clase 1, aquí «fuera de especificación». Su pérdida es la entropía cruzada para
dos clases, `'binary_crossentropy'`.

Construye y entrena esta red:

| Qué | Cómo |
|---|---|
| Semilla | `keras.utils.set_random_seed(0)` antes de construir la red, como en el satélite |
| Entrada | `keras.Input(shape=(9,))`: las 9 entradas estandarizadas |
| Capa oculta | `Dense` de 16 neuronas con activación `'relu'` |
| Capa de salida | `Dense` de 1 neurona con activación `'sigmoid'` |
| Compilación | optimizador `'adam'` y pérdida `'binary_crossentropy'` |
| Entrenamiento | con `X_ajuste` e `y_ajuste_clase`, validando con `X_validacion` e `y_validacion_clase`; lotes de 32 y un máximo de 200 épocas |
| Parada temprana | `EarlyStopping` sobre la pérdida de validación (`'val_loss'`), con paciencia 10 y recuperando los pesos de la mejor época |

Variables que tiene que dejar preparadas tu código:

| Variable | Qué contiene |
|---|---|
| `red_clase` | la red entrenada |
| `historial_clase` | lo que devuelve `fit`: la pérdida de ajuste y de validación en cada época |
| `probabilidad_red` | probabilidad de «fuera de especificación» que da la red en cada hora del test: un vector de 820 valores |

In [ ]:
# === Tu código ===
# Entrena la red de la tabla (tarda unos segundos):
# 1) Fija la semilla con keras.utils.set_random_seed(0).
# 2) Construye red_clase con keras.Sequential.
# 3) Compílala.
# 4) Crea la parada temprana con keras.callbacks.EarlyStopping.
# 5) Entrénala con fit (usa verbose=0) y guarda lo que devuelve en historial_clase.
# 6) Calcula la salida de la red para X_test con red_clase.predict(X_test, verbose=0).
#    Es una tabla de 820 filas y 1 columna: guarda esa columna en probabilidad_red con [:, 0].



**Comprueba:**

In [ ]:
capas_clase = red_clase.layers
assert len(capas_clase) == 2, "La red debe tener dos capas Dense: la oculta y la de salida"
assert capas_clase[0].units == 16, "La capa oculta debe tener 16 neuronas"
assert capas_clase[1].units == 1, "La capa de salida debe tener 1 neurona"
assert capas_clase[1].activation.__name__ == 'sigmoid', "La capa de salida debe usar la activación 'sigmoid'"
assert red_clase.loss == 'binary_crossentropy', "Compila la red con la pérdida 'binary_crossentropy'"

n_epocas_comprobacion = len(historial_clase.history['loss'])
assert n_epocas_comprobacion < 200, "La red ha llegado a 200 épocas: ¿has pasado la parada temprana en callbacks?"

assert probabilidad_red.shape == (820,), "probabilidad_red debe ser un vector de 820 valores: usa [:, 0]"
auc_comprobacion = roc_auc_score(y_test_clase, probabilidad_red)
assert auc_comprobacion > 0.75, "El AUC de la red es muy bajo: revisa las entradas, las salidas con las que entrenas y la pérdida"
print("Correcto")

### Curva de pérdida, matrices de confusión y métricas

Ejecuta las tres celdas sin cambiar nada. La primera dibuja la pérdida de ajuste y de validación en cada
época y marca la época con la menor pérdida de validación. La segunda muestra las matrices de
confusión de la logística y de la red en el test, con el umbral 0,5. La tercera compara las métricas del
test.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `n_epocas_clase` | épocas que entrenó la red antes de detenerse |
| `mejor_epoca_clase` | época con la menor pérdida de validación |
| `prediccion_red` | 1 si `probabilidad_red` llega a 0,5; 0 si no |

In [ ]:
perdidas_ajuste_clase = historial_clase.history['loss']
perdidas_validacion_clase = historial_clase.history['val_loss']
n_epocas_clase = len(perdidas_ajuste_clase)
posicion_mejor_clase = np.argmin(perdidas_validacion_clase)
mejor_epoca_clase = posicion_mejor_clase + 1
epocas_clase = np.arange(1, n_epocas_clase + 1)
print(f"Se detuvo en la época {n_epocas_clase}; la menor pérdida de validación fue la de la época {mejor_epoca_clase}.")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(epocas_clase, perdidas_ajuste_clase, label='ajuste')
ax.plot(epocas_clase, perdidas_validacion_clase, label='validación')
ax.axvline(mejor_epoca_clase, color='gray', ls='--', lw=1, label='menor pérdida de validación')
ax.set_xlabel('época')
ax.set_ylabel('pérdida (entropía cruzada)')
ax.set_title('Red de clasificación')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
supera_umbral_red = probabilidad_red >= 0.5
prediccion_red = supera_umbral_red.astype(int)

matriz_logistica = confusion_matrix(y_test_clase, prediccion_logistica)
matriz_red = confusion_matrix(y_test_clase, prediccion_red)

fig, ax = plt.subplots(1, 2, figsize=(9, 3.8))
grafico_logistica = ConfusionMatrixDisplay(matriz_logistica, display_labels=["dentro", "fuera"])
grafico_logistica.plot(ax=ax[0], cmap="Blues", colorbar=False)
ax[0].set_title("Regresión logística")
grafico_red = ConfusionMatrixDisplay(matriz_red, display_labels=["dentro", "fuera"])
grafico_red.plot(ax=ax[1], cmap="Blues", colorbar=False)
ax[1].set_title("Red de clasificación")
for eje in ax:
    eje.set_xlabel("clase predicha")
    eje.set_ylabel("clase real")
fig.suptitle("Matrices de confusión en el test (umbral 0,5)")
plt.tight_layout()
plt.show()

In [ ]:
exactitud_base = accuracy_score(y_test_clase, prediccion_base_clase)
recall_base = recall_score(y_test_clase, prediccion_base_clase)

exactitud_logistica = accuracy_score(y_test_clase, prediccion_logistica)
precision_logistica = precision_score(y_test_clase, prediccion_logistica)
recall_logistica = recall_score(y_test_clase, prediccion_logistica)
f1_logistica = f1_score(y_test_clase, prediccion_logistica)
auc_logistica = roc_auc_score(y_test_clase, probabilidad_logistica)

exactitud_red = accuracy_score(y_test_clase, prediccion_red)
precision_red = precision_score(y_test_clase, prediccion_red)
recall_red = recall_score(y_test_clase, prediccion_red)
f1_red = f1_score(y_test_clase, prediccion_red)
auc_red = roc_auc_score(y_test_clase, probabilidad_red)

print("Métricas en el test (umbral 0,5):")
print(f"{'':12s} {'exactitud':>9s} {'precisión':>9s} {'recall':>7s} {'F1':>6s} {'AUC':>6s}")
print(f"{'línea base':12s} {exactitud_base:9.3f} {'—':>9s} {recall_base:7.3f} {0:6.3f} {0.5:6.3f}")
print(f"{'logística':12s} {exactitud_logistica:9.3f} {precision_logistica:9.3f} {recall_logistica:7.3f} {f1_logistica:6.3f} {auc_logistica:6.3f}")
print(f"{'red':12s} {exactitud_red:9.3f} {precision_red:9.3f} {recall_red:7.3f} {f1_red:6.3f} {auc_red:6.3f}")

**Pregunta 2.1.** ¿Bate la red a la línea base y a la regresión logística? Compáralas en exactitud,
recall y AUC. Con la matriz de confusión de la red, ¿de cuántas de las horas que de verdad salieron
fuera de especificación avisa, cuántas se le escapan y cuántas falsas alarmas da?

**Tu respuesta:** *escribe aquí*

**Pregunta 2.2.** En la curva de pérdida, ¿en qué época se detuvo el entrenamiento y de qué época son
los pesos con los que se ha quedado la red? ¿Por qué esa época se elige con la validación y no con el
test?

**Tu respuesta:** *escribe aquí*

## 3. Red de regresión: estimar la sílice

Ahora la red estima el % de sílice de cada hora. Como en la regresión del satélite, las entradas y la
capa oculta no cambian; cambian la capa de salida y la pérdida. Construye y entrena una red **igual que
la de la sección 2**, con la misma semilla, la misma capa oculta y la misma parada temprana, salvo en
esto:

| | Sección 2 (clasificación) | Esta sección (regresión) |
|---|---|---|
| Capa de salida | `Dense(1, activation='sigmoid')` | `Dense(1)`: una neurona, sin activación |
| Pérdida | `'binary_crossentropy'` | `'mse'`: error cuadrático medio |
| Salidas para entrenar y validar | `y_ajuste_clase`, `y_validacion_clase` | `y_ajuste_silice`, `y_validacion_silice` |

Variables que tiene que dejar preparadas tu código:

| Variable | Qué contiene |
|---|---|
| `red_silice` | la red entrenada |
| `historial_silice` | lo que devuelve `fit` |
| `prediccion_red_silice` | sílice que estima la red en cada hora del test: un vector de 820 valores |

In [ ]:
# === Tu código ===
# Los mismos pasos que en la sección 2, con los cambios de la tabla (tarda unos segundos).
# Deja preparadas red_silice, historial_silice y prediccion_red_silice.



**Comprueba:**

In [ ]:
capas_silice = red_silice.layers
assert len(capas_silice) == 2, "La red debe tener dos capas Dense: la oculta y la de salida"
assert capas_silice[0].units == 16, "La capa oculta debe tener 16 neuronas"
assert capas_silice[1].units == 1, "La capa de salida debe tener 1 neurona"
assert capas_silice[1].activation.__name__ == 'linear', "La capa de salida de regresión no lleva activación"
assert red_silice.loss == 'mse', "Compila la red con la pérdida 'mse'"

n_epocas_comprobacion = len(historial_silice.history['loss'])
assert n_epocas_comprobacion < 200, "La red ha llegado a 200 épocas: ¿has pasado la parada temprana en callbacks?"

assert prediccion_red_silice.shape == (820,), "prediccion_red_silice debe ser un vector de 820 valores: usa [:, 0]"
r2_comprobacion = r2_score(y_test_silice, prediccion_red_silice)
assert r2_comprobacion > 0.3, "El R² de la red es muy bajo: revisa las salidas con las que entrenas y la pérdida"
print("Correcto")

### Curva de pérdida, métricas y sílice real frente a estimada

Ejecuta las tres celdas sin cambiar nada. La primera dibuja la curva de pérdida, como en la sección 2,
pero en escala logarítmica: la pérdida empieza muy alta y, en escala normal, no se vería el final. La
segunda compara las métricas del test con las de la Semana 2: MAE y RMSE en % de sílice (cuanto más
pequeños, mejor) y R² (1 es una estimación perfecta; 0, lo mismo que predecir siempre la media). La
tercera dibuja, para cada hora del test, la sílice estimada frente a la real.

In [ ]:
perdidas_ajuste_silice = historial_silice.history['loss']
perdidas_validacion_silice = historial_silice.history['val_loss']
n_epocas_silice = len(perdidas_ajuste_silice)
posicion_mejor_silice = np.argmin(perdidas_validacion_silice)
mejor_epoca_silice = posicion_mejor_silice + 1
epocas_silice = np.arange(1, n_epocas_silice + 1)
print(f"Se detuvo en la época {n_epocas_silice}; la menor pérdida de validación fue la de la época {mejor_epoca_silice}.")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(epocas_silice, perdidas_ajuste_silice, label='ajuste')
ax.plot(epocas_silice, perdidas_validacion_silice, label='validación')
ax.axvline(mejor_epoca_silice, color='gray', ls='--', lw=1, label='menor pérdida de validación')
ax.set_xlabel('época')
ax.set_yscale('log')
ax.set_ylabel('pérdida (error cuadrático medio)')
ax.set_title('Red de regresión (escala logarítmica)')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
mae_base_silice = mean_absolute_error(y_test_silice, prediccion_base_silice)
rmse_base_silice = root_mean_squared_error(y_test_silice, prediccion_base_silice)
r2_base_silice = r2_score(y_test_silice, prediccion_base_silice)

mae_lineal = mean_absolute_error(y_test_silice, prediccion_lineal)
rmse_lineal = root_mean_squared_error(y_test_silice, prediccion_lineal)
r2_lineal = r2_score(y_test_silice, prediccion_lineal)

mae_red_silice = mean_absolute_error(y_test_silice, prediccion_red_silice)
rmse_red_silice = root_mean_squared_error(y_test_silice, prediccion_red_silice)
r2_red_silice = r2_score(y_test_silice, prediccion_red_silice)

print("Métricas en el test:")
print(f"{'':12s} {'MAE':>6s} {'RMSE':>6s} {'R²':>7s}")
print(f"{'línea base':12s} {mae_base_silice:6.3f} {rmse_base_silice:6.3f} {r2_base_silice:+7.3f}")
print(f"{'lineal':12s} {mae_lineal:6.3f} {rmse_lineal:6.3f} {r2_lineal:+7.3f}")
print(f"{'red':12s} {mae_red_silice:6.3f} {rmse_red_silice:6.3f} {r2_red_silice:+7.3f}")

In [ ]:
limites = [0.5, 6]

fig, ax = plt.subplots(1, 2, figsize=(10, 4.2), sharex=True, sharey=True)
ax[0].scatter(y_test_silice, prediccion_lineal, s=6, alpha=0.4)
ax[0].set_title(f"Regresión lineal (R² = {r2_lineal:.2f})")
ax[1].scatter(y_test_silice, prediccion_red_silice, s=6, alpha=0.4, color='C1')
ax[1].set_title(f"Red de regresión (R² = {r2_red_silice:.2f})")
for eje in ax:
    eje.plot(limites, limites, color='C3', lw=1, label='estimación perfecta')
    eje.set_xlabel('sílice real (%)')
    eje.grid(alpha=0.3)
    eje.legend()
ax[0].set_ylabel('sílice estimada (%)')
fig.suptitle("Horas del test: sílice estimada frente a la real")
plt.tight_layout()
plt.show()

**Pregunta 3.1.** ¿Bate la red de regresión a la línea base y a la regresión lineal? ¿Por cuánto, en % de
sílice (diferencia de MAE)? ¿Qué significa su R² del test?

**Tu respuesta:** *escribe aquí*

> **Antes de entregar:** reinicia el kernel y ejecuta todo (en VS Code, «Restart» y luego «Run All»;
> en Colab, «Entorno de ejecución → Reiniciar y ejecutar todo»). Comprueba que no hay errores y que todas las celdas «Comprueba» dicen «Correcto». Guarda y sube el `.ipynb` al
> Campus Virtual.